# Phase 1 — Environment Check

Verifica librerías, credenciales y crea la estructura de carpetas.
Ejecuta las celdas en orden. Cada sección debe terminar sin errores antes de pasar a los notebooks de descarga.

## 1. Librerías instaladas

In [1]:
import sys
import importlib

required = [
    "torch", "torchvision", "torchgeo",
    "segmentation_models_pytorch",
    "rasterio", "geopandas", "numpy", "matplotlib",
    "pystac_client", "requests", "dotenv", "tqdm", "cdsapi"
]

print(f"Python {sys.version}\n")
missing = []
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'ok')
        print(f"  OK      {pkg:<35} {version}")
    except ImportError:
        print(f"  MISSING {pkg}")
        missing.append(pkg)

if missing:
    print(f"\nInstala los faltantes: pip install {' '.join(missing)}")
else:
    print("\nTodas las librerías presentes.")

Python 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:36:00) [MSC v.1944 64 bit (AMD64)]

  OK      torch                               2.12.0+cpu
  OK      torchvision                         0.27.0+cpu
  OK      torchgeo                            0.8.1
  OK      segmentation_models_pytorch         0.5.0
  OK      rasterio                            1.4.4
  OK      geopandas                           1.1.3
  OK      numpy                               2.2.6
  OK      matplotlib                          3.9.1
  OK      pystac_client                       0.9.0
  OK      requests                            2.34.2
  OK      dotenv                              ok
  OK      tqdm                                4.67.3
  OK      cdsapi                              ok

Todas las librerías presentes.


## 2. Credenciales (.env)

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("..") / ".env"
assert env_path.exists(), f".env no encontrado en {env_path.resolve()}"
load_dotenv(dotenv_path=env_path)

keys = ["CDSE_USER", "CDSE_PASSWORD", "FIRMS_API_KEY", "CDS_URL", "CDS_KEY"]
for k in keys:
    v = os.getenv(k)
    if v and v != f"your_{k.lower()}":
        print(f"  OK  {k}: {v[:6]}...")
    else:
        print(f"  FALTA o placeholder: {k}")

  OK  CDSE_USER: agusti...
  OK  CDSE_PASSWORD: Polyte...
  OK  FIRMS_API_KEY: 63fb7b...
  OK  CDS_URL: https:...
  OK  CDS_KEY: c3ffc2...


## 3. Test CDSE — token OAuth2

In [3]:
import requests

def get_cdse_token(user, password):
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={
            "client_id": "cdse-public",
            "grant_type": "password",
            "username": user,
            "password": password,
        },
        timeout=30
    )
    r.raise_for_status()
    return r.json()["access_token"]

token = get_cdse_token(os.getenv("CDSE_USER"), os.getenv("CDSE_PASSWORD"))
print(f"CDSE OK — token: {token[:40]}...")

CDSE OK — token: eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwi...


## 4. Test CDSE STAC — búsqueda de catálogo

In [4]:
import pystac_client

catalog = pystac_client.Client.open(
    "https://catalogue.dataspace.copernicus.eu/stac"
)

# Búsqueda mínima: 1 tile sobre Corrientes en enero 2022
search = catalog.search(
    collections=["SENTINEL-2"],
    bbox=[-58.5, -28.5, -57.5, -27.5],
    datetime="2022-01-10T00:00:00Z/2022-01-15T23:59:59Z",
    query={"eo:cloud_cover": {"lt": 30}, "s2:product_type": {"eq": "S2MSI2A"}},
    max_items=3
)

items = list(search.items())
print(f"Items encontrados: {len(items)}")
if items:
    item = items[0]
    print(f"\nPrimer item: {item.id}")
    print(f"Fecha: {item.datetime}")
    print(f"Cloud cover: {item.properties.get('eo:cloud_cover')}%")
    print(f"\nAssets disponibles: {list(item.assets.keys())}")

Items encontrados: 0


## 5. Test NASA FIRMS

In [5]:
import requests

FIRMS_KEY = os.getenv("FIRMS_API_KEY")

# Verifica que el key es válido — 1 día, bbox pequeño
url = (
    f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{FIRMS_KEY}/VIIRS_SNPP_SP/-58.5,-28.5,-57.5,-27.5/1/2022-01-15"
)

r = requests.get(url, timeout=30)
r.raise_for_status()

lines = r.text.strip().split("\n")
print(f"FIRMS OK — {len(lines) - 1} detecciones devueltas (1 día, bbox pequeño)")
print(f"Header: {lines[0]}")

FIRMS OK — 111 detecciones devueltas (1 día, bbox pequeño)
Header: latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type


## 6. Test CDS — cdsapi

In [6]:
import cdsapi

# Lee automáticamente C:\Users\<user>\.cdsapirc
client = cdsapi.Client(quiet=True)
print(f"CDS OK — URL: {client.url}")

CDS OK — URL: https://cds.climate.copernicus.eu/api


## 7. Crear estructura de carpetas de datos

In [7]:
from pathlib import Path

project_root = Path("..").resolve()

dirs = [
    project_root / "data" / "sentinel2" / "raw",
    project_root / "data" / "firms",
    project_root / "data" / "era5",
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  OK  {d}")

print("\nEstructura de datos lista.")

  OK  D:\GeoAI\wildfire-spread\data\sentinel2\raw
  OK  D:\GeoAI\wildfire-spread\data\firms
  OK  D:\GeoAI\wildfire-spread\data\era5

Estructura de datos lista.


---
**Todos los tests OK → continuar con `01_download_sentinel2.ipynb`**